# 上傳ZIP檔案到watsonx.ai並建立批次評分作業

#### 特點
- 上傳ZIP檔案到watsonx.ai
- 建立批次評分作業
- 監控作業狀態
- 取得評分結果

In [3]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json
import time
import os
import getpass

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

print("✅ 載入必要套件")

✅ 載入必要套件


## 配置參數

In [4]:
# watsonx.ai 認證資訊（從根目錄的.env載入）
CLOUD_API_KEY = getpass.getpass("Enter your watsonx.ai API key: ")  # IBM Cloud API Key
SPACE_ID = "165e735f-11f6-4879-a6fd-99cfecf44f37"  # watsonx.ai 部署空間ID
WATSONX_URL = "https://us-south.ml.cloud.ibm.com"  # watsonx.ai URL

# 部署資訊
DEPLOYMENT_ID = "6b1cc9f1-347c-4a43-aa94-67115b5c26b4"  # 批次部署ID

# 輸入檔案 - 自動找到最新的ZIP檔案
DATA_DIR = Path("data/zipped")
zip_files = sorted(DATA_DIR.glob("batch_input_*.zip"))
if not zip_files:
    raise FileNotFoundError(f"在 {DATA_DIR} 中找不到batch_input_*.zip檔案")
INPUT_ZIP = zip_files[-1]  # 使用最新的ZIP檔案

# 監控參數
CHECK_INTERVAL = 2  # 檢查作業狀態的間隔（秒）

print(f"watsonx.ai URL: {WATSONX_URL}")
print(f"空間ID: {SPACE_ID}")
print(f"部署ID: {DEPLOYMENT_ID}")
print(f"輸入ZIP: {INPUT_ZIP}")

Enter your watsonx.ai API key:  ········


watsonx.ai URL: https://us-south.ml.cloud.ibm.com
空間ID: 165e735f-11f6-4879-a6fd-99cfecf44f37
部署ID: 6b1cc9f1-347c-4a43-aa94-67115b5c26b4
輸入ZIP: data/zipped/batch_input_20260412_223409.zip


## 初始化watsonx.ai客戶端

In [5]:
try:
    from ibm_watsonx_ai import APIClient
    from ibm_watsonx_ai.metanames import DeploymentMetaNames
    
    print("初始化watsonx.ai客戶端...")
    
    # 設定認證資訊
    credentials = {
        "url": WATSONX_URL,
        "apikey": CLOUD_API_KEY
    }
    
    # 建立客戶端
    client = APIClient(credentials)
    
    # 設定專案或空間
    # if PROJECT_ID:
    #     client.set.default_project(PROJECT_ID)
    #     print(f"✅ 使用專案ID: {PROJECT_ID}")
    # else:
    #     raise ValueError("必須在.env中設定 PROJECT_ID")
    if SPACE_ID:
        client.set.default_space(SPACE_ID)
        print(f"✅ 使用空間ID: {SPACE_ID}")
    else:
        raise ValueError("必須在.env中設定 SPACE_ID")
    
    print("✅ watsonx.ai客戶端初始化成功")
    
except ImportError:
    print("❌ 找不到 ibm-watsonx-ai 套件")
    print("請執行: pip install ibm-watsonx-ai")
    raise
except Exception as e:
    print(f"❌ 初始化watsonx.ai客戶端失敗: {e}")
    raise

初始化watsonx.ai客戶端...
✅ 使用空間ID: 165e735f-11f6-4879-a6fd-99cfecf44f37
✅ watsonx.ai客戶端初始化成功


## 檢查輸入檔案

In [6]:
# 檢查ZIP檔案是否存在
if not INPUT_ZIP.exists():
    raise FileNotFoundError(f"找不到ZIP檔案: {INPUT_ZIP}")

file_size = INPUT_ZIP.stat().st_size
print(f"ZIP檔案: {INPUT_ZIP}")
print(f"檔案大小: {file_size / (1024**2):.2f} MB ({file_size / (1024**3):.2f} GB)")

ZIP檔案: data/zipped/batch_input_20260412_223409.zip
檔案大小: 14.62 MB (0.01 GB)


## 上傳ZIP檔案到watsonx.ai

In [7]:
print("\n開始上傳檔案到watsonx.ai...")
upload_start_time = datetime.now(TZ_UTC8)

try:
    # 上傳檔案到資料資產
    asset_name = f"batch_input_{datetime.now(TZ_UTC8).strftime('%Y%m%d_%H%M%S')}.zip"
    
    print(f"資產名稱: {asset_name}")
    print("上傳中...（這可能需要幾分鐘，取決於檔案大小和網路速度）")
    
    asset_details = client.data_assets.create(
        name=asset_name,
        file_path=str(INPUT_ZIP)
    )
    
    data_asset_id = client.data_assets.get_id(asset_details)
    
    upload_end_time = datetime.now(TZ_UTC8)
    upload_duration = (upload_end_time - upload_start_time).total_seconds()
    
    print(f"\n✅ 檔案上傳成功！")
    print(f"資產ID: {data_asset_id}")
    print(f"上傳時間: {upload_duration:.2f} 秒 ({upload_duration/60:.2f} 分鐘)")
    print(f"上傳速度: {file_size / upload_duration / (1024**2):.2f} MB/秒")
    
except Exception as e:
    print(f"❌ 上傳檔案失敗: {e}")
    raise


開始上傳檔案到watsonx.ai...
資產名稱: batch_input_20260412_223530.zip
上傳中...（這可能需要幾分鐘，取決於檔案大小和網路速度）
Creating data asset...
SUCCESS

✅ 檔案上傳成功！
資產ID: 7aa952ce-ef0a-4b8a-9d3d-0a748e908e71
上傳時間: 1.46 秒 (0.02 分鐘)
上傳速度: 9.99 MB/秒


## 建立批次評分作業

In [8]:
if data_asset_id:
    # 取得 data asset 的 href
    asset_details = client.data_assets.get_details(data_asset_id)
    asset_href = asset_details['metadata']['href']
    
    # 準備輸出檔案名稱
    output_timestamp = datetime.now(TZ_UTC8).strftime('%Y%m%d_%H%M%S')
    output_name = f"batch_scoring_output_{output_timestamp}.json"
    
    # 準備 job payload（引用 data asset）
    job_payload = {
        client.deployments.ScoringMetaNames.INPUT_DATA_REFERENCES: [{
            'type': 'data_asset',
            'location': {
                'id': data_asset_id,
                'href': asset_href
            }
        }],
        client.deployments.ScoringMetaNames.OUTPUT_DATA_REFERENCE: {
            'type': 'data_asset',
            'location': {
                'name': output_name
            }
        }
    }
    
    print("="*80)
    print("CREATING BATCH SCORING JOB")
    print("="*80)
    print(f"📁 Input data asset ID: {data_asset_id}")
    print(f"📁 Input data asset href: {asset_href}")
    print(f"📁 Output data asset name: {output_name}")
    print()
    
    try:
        # 建立 batch job
        job_details = client.deployments.create_job(DEPLOYMENT_ID, job_payload)
        job_id = client.deployments.get_job_uid(job_details)
        
        print(f"✅ Batch job created successfully!")
        print(f"✅ Job ID: {job_id}")
        print(f"\n⏳ Job is now queued for execution...")
        print(f"📁 Output will be saved as a new data asset in the Space")
        
    except Exception as e:
        print(f"❌ Error creating batch job: {str(e)}")
        print("\n💡 Troubleshooting tips:")
        print("   1. Verify deployment is in 'ready' state")
        print("   2. Check data asset ID is correct")
        print("   3. Ensure ZIP contains valid JSON format matching model input")
        job_id = None
else:
    print("❌ No data asset ID available. Please upload ZIP first.")
    job_id = None

CREATING BATCH SCORING JOB
📁 Input data asset ID: 7aa952ce-ef0a-4b8a-9d3d-0a748e908e71
📁 Input data asset href: /v2/assets/7aa952ce-ef0a-4b8a-9d3d-0a748e908e71?space_id=165e735f-11f6-4879-a6fd-99cfecf44f37
📁 Output data asset name: batch_scoring_output_20260412_223534.json

Note: The `/text/generation` and `/text/generation_stream` endpoints will be deprecated in an upcoming release and removed in a future release.
✅ Batch job created successfully!
✅ Job ID: 73176e3c-b72d-4306-94d5-d4ceb09e934a

⏳ Job is now queued for execution...
📁 Output will be saved as a new data asset in the Space


## 監控作業狀態

In [9]:
if job_id:
    print("="*80)
    print("MONITORING JOB STATUS")
    print("="*80)
    print(f"Job ID: {job_id}")
    print("\n⏳ Waiting for job to complete...\n")
    
    # 監控 job 執行狀態
    check_count = 0
    state = None  # Initialize state variable
    job_status = None  # Initialize job_status variable
    while True:
        try:
            job_status = client.deployments.get_job_details(job_id)
            state = job_status['entity']['scoring']['status']['state']
            
            # 顯示當前狀態
            check_count += 1
            timestamp = datetime.now(TZ_UTC8).strftime('%Y-%m-%d %H:%M:%S')
            print(f"[{timestamp}] Check #{check_count}: Job status = {state}")
            
            # 檢查是否完成
            if state in ['completed', 'failed', 'canceled', 'error']:
                break
            
            # 等待 CHECK_INTERVAL 秒後再次檢查
            time.sleep(CHECK_INTERVAL)
            
        except Exception as e:
            print(f"❌ Error checking job status: {str(e)}")
            break
    
    # 顯示最終狀態
    print("\n" + "="*80)
    if state is None:
        print("❌ BATCH SCORING FAILED - Unable to retrieve job status")
        output_asset_id = None
    elif state == 'completed':
        print("✅ BATCH SCORING COMPLETED SUCCESSFULLY!")
        
        # 取得輸出 data asset ID
        if job_status and 'scoring' in job_status.get('entity', {}) and 'output_data_references' in job_status['entity']['scoring']:
            output_refs = job_status['entity']['scoring']['output_data_references']
            if output_refs and len(output_refs) > 0:
                output_asset_id = output_refs[0]['location'].get('id')
                print(f"📁 Output data asset ID: {output_asset_id}")
            else:
                output_asset_id = None
        else:
            output_asset_id = None
            
    elif state == 'failed':
        print("❌ BATCH SCORING FAILED!")
        if job_status and 'message' in job_status.get('entity', {}).get('scoring', {}).get('status', {}):
            print(f"   Error message: {job_status['entity']['scoring']['status']['message']}")
        output_asset_id = None
    elif state == 'canceled':
        print("⚠️  BATCH SCORING WAS CANCELED")
        output_asset_id = None
    elif state:
        print(f"⚠️  BATCH SCORING ENDED WITH STATUS: {state}")
        output_asset_id = None
    else:
        print("⚠️  BATCH SCORING STATUS UNKNOWN (monitoring interrupted)")
        output_asset_id = None
    print("="*80)
else:
    print("❌ No job ID available. Please create a job first.")
    state = None
    output_asset_id = None

MONITORING JOB STATUS
Job ID: 73176e3c-b72d-4306-94d5-d4ceb09e934a

⏳ Waiting for job to complete...

[2026-04-12 22:35:42] Check #1: Job status = queued
[2026-04-12 22:35:44] Check #2: Job status = queued
[2026-04-12 22:35:47] Check #3: Job status = queued
[2026-04-12 22:35:50] Check #4: Job status = queued
[2026-04-12 22:35:52] Check #5: Job status = queued
[2026-04-12 22:35:54] Check #6: Job status = queued
[2026-04-12 22:35:56] Check #7: Job status = queued
[2026-04-12 22:35:58] Check #8: Job status = queued
[2026-04-12 22:36:00] Check #9: Job status = queued
[2026-04-12 22:36:02] Check #10: Job status = queued
[2026-04-12 22:36:05] Check #11: Job status = queued
[2026-04-12 22:36:07] Check #12: Job status = running
[2026-04-12 22:36:09] Check #13: Job status = running
[2026-04-12 22:36:11] Check #14: Job status = running
[2026-04-12 22:36:13] Check #15: Job status = running
[2026-04-12 22:36:15] Check #16: Job status = running
[2026-04-12 22:36:17] Check #17: Job status = running


## 取得作業詳細資訊

In [10]:
# 取得完整的作業詳細資訊
print("\n取得作業詳細資訊...")
final_job_details = client.deployments.get_job_details(job_id)

print("\n作業詳細資訊:")
print(json.dumps(final_job_details, indent=2, ensure_ascii=False))


取得作業詳細資訊...

作業詳細資訊:
{
  "entity": {
    "deployment": {
      "id": "6b1cc9f1-347c-4a43-aa94-67115b5c26b4"
    },
    "platform_job": {
      "job_id": "146dab39-8488-4419-8132-e55aa70f4e29",
      "run_id": "73176e3c-b72d-4306-94d5-d4ceb09e934a"
    },
    "scoring": {
      "input_data_references": [
        {
          "connection": {},
          "location": {
            "href": "/v2/assets/7aa952ce-ef0a-4b8a-9d3d-0a748e908e71?space_id=165e735f-11f6-4879-a6fd-99cfecf44f37",
            "id": "7aa952ce-ef0a-4b8a-9d3d-0a748e908e71"
          },
          "type": "data_asset"
        }
      ],
      "output_data_reference": {
        "connection": {},
        "location": {
          "href": "/v2/assets/ddc9ccf5-93cf-4332-bf43-1f470ae12b6f?space_id=165e735f-11f6-4879-a6fd-99cfecf44f37",
          "name": "batch_scoring_output_20260412_223534.json"
        },
        "type": "data_asset"
      },
      "status": {
        "completed_at": "2026-04-12T14:36:22.840456Z",
        "runnin

## 儲存統計資訊

In [ ]:
# 儲存統計資訊
timestamp = datetime.now(TZ_UTC8).strftime('%Y%m%d_%H%M%S')
stats_file = Path("logs") / f"job_stats_{timestamp}.json"
stats_file.parent.mkdir(parents=True, exist_ok=True)

stats_data = {
    "deployment_id": DEPLOYMENT_ID,
    "job_id": job_id,
    "asset_id": asset_id,
    "input_zip": str(INPUT_ZIP),
    "file_size_bytes": file_size,
    "file_size_mb": file_size / (1024**2),
    "file_size_gb": file_size / (1024**3),
    "upload_duration_seconds": upload_duration,
    "upload_speed_mbps": file_size / upload_duration / (1024**2),
    "job_duration_seconds": job_duration if 'job_duration' in locals() else None,
    "job_status": status,
    "upload_start_time": upload_start_time.isoformat(),
    "upload_end_time": upload_end_time.isoformat(),
    "job_start_time": job_start_time.isoformat(),
    "job_end_time": job_end_time.isoformat() if 'job_end_time' in locals() else None,
}

with open(stats_file, 'w') as f:
    json.dump(stats_data, f, indent=2, ensure_ascii=False)

print(f"\n統計資訊已儲存至: {stats_file}")

## 總結

In [ ]:
print("\n" + "=" * 60)
print("批次預測流程完成總結")
print("=" * 60)
print(f"✅ 上傳檔案: {INPUT_ZIP.name}")
print(f"✅ 檔案大小: {file_size / (1024**3):.2f} GB")
print(f"✅ 上傳時間: {upload_duration/60:.2f} 分鐘")
print(f"✅ 資產ID: {asset_id}")
print(f"✅ 作業ID: {job_id}")
print(f"✅ 作業狀態: {status}")
if 'job_duration' in locals():
    print(f"✅ 作業執行時間: {job_duration/60:.2f} 分鐘")
print("=" * 60)
print("\n🎉 完整流程執行完成！")